In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

# nb_img_05b: MLP Pseudo-Labeling — Config B

Self-training MLP on frozen CLIP embeddings.  
Config B: `threshold=0.85`, `hidden_sizes=(256, 128, 64)`

In [2]:
import numpy as np
import pandas as pd
import json
from sklearn.metrics import f1_score
from utils.image_config import IMAGE_MANIFEST_PATH, IMAGE_EMBEDDINGS_PATH, IMAGE_CLASS_NAMES, IMAGE_SEED, RESULTS_DIR
from utils.image_pseudo_labeling import pseudo_label_loop

manifest = pd.read_csv(IMAGE_MANIFEST_PATH)
embeddings = np.load(IMAGE_EMBEDDINGS_PATH)

seed_mask = manifest["is_seed"].to_numpy().astype(bool)
seed_labels = manifest.loc[seed_mask, "label_idx"].to_numpy()
true_labels = manifest["label_idx"].to_numpy()

print(f"Seed: {seed_mask.sum()}  Unlabeled: {(~seed_mask).sum()}")

Seed: 275  Unlabeled: 5225


In [3]:
pred_labels, history = pseudo_label_loop(
    embeddings=embeddings,
    seed_mask=seed_mask,
    seed_labels=seed_labels,
    hidden_sizes=(256, 128, 64),
    threshold=0.85,
    max_iters=10,
    seed=IMAGE_SEED,
    n_classes=len(IMAGE_CLASS_NAMES),
)
print("History:", history)
final_coverage = (pred_labels != -1).mean()
label_acc = (pred_labels == true_labels).mean()
label_f1 = f1_score(true_labels, pred_labels, average="macro", zero_division=0)
print(f"Config B — ACC: {label_acc:.4f}  F1: {label_f1:.4f}  Coverage: {final_coverage:.1%}")

  Iter 0: +5075 pseudo-labels  coverage=97.3%


  Iter 1: +99 pseudo-labels  coverage=99.1%


  Iter 2: +20 pseudo-labels  coverage=99.4%


  Iter 3: +10 pseudo-labels  coverage=99.6%


  Iter 4: +11 pseudo-labels  coverage=99.8%


...

  Iter 6: +3 pseudo-labels  coverage=99.9%


  Iter 7: +1 pseudo-labels  coverage=99.9%


  Iter 8: +1 pseudo-labels  coverage=99.9%


  Iter 9: +0 pseudo-labels  coverage=99.9%


History: [{'iteration': 0, 'new_pseudo_labels': 5075, 'cumulative_coverage': 0.9727272727272728}, {'iteration': 1, 'new_pseudo_labels': 99, 'cumulative_coverage': 0.9907272727272727}, {'iteration': 2, 'new_pseudo_labels': 20, 'cumulative_coverage': 0.9943636363636363}, {'iteration': 3, 'new_pseudo_labels': 10, 'cumulative_coverage': 0.9961818181818182}, {'iteration': 4, 'new_pseudo_labels': 11, 'cumulative_coverage': 0.9981818181818182}, {'iteration': 5, 'new_pseudo_labels': 1, 'cumulative_coverage': 0.9983636363636363}, {'iteration': 6, 'new_pseudo_labels': 3, 'cumulative_coverage': 0.9989090909090909}, {'iteration': 7, 'new_pseudo_labels': 1, 'cumulative_coverage': 0.9990909090909091}, {'iteration': 8, 'new_pseudo_labels': 1, 'cumulative_coverage': 0.9992727272727273}, {'iteration': 9, 'new_pseudo_labels': 0, 'cumulative_coverage': 0.9992727272727273}]
Config B — ACC: 0.9915  F1: 0.9915  Coverage: 100.0%


In [4]:
out_df = manifest[["filepath", "true_label"]].copy()
out_df["predicted_label"] = [IMAGE_CLASS_NAMES[p] for p in pred_labels]
RESULTS_DIR.mkdir(exist_ok=True)
out_df.to_csv(RESULTS_DIR / "image_full_labels_pseudo_labeling_B.csv", index=False)

metrics = {
    "config": "B",
    "hidden_sizes": [256, 128, 64],
    "threshold": 0.85,
    "label_acc": float(label_acc),
    "label_f1": float(label_f1),
    "coverage": float(final_coverage),
    "history": history,
}
with open(RESULTS_DIR / "image_metrics_pseudo_labeling_B.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)

{'config': 'B', 'hidden_sizes': [256, 128, 64], 'threshold': 0.85, 'label_acc': 0.9914545454545455, 'label_f1': 0.9914600422076022, 'coverage': 1.0, 'history': [{'iteration': 0, 'new_pseudo_labels': 5075, 'cumulative_coverage': 0.9727272727272728}, {'iteration': 1, 'new_pseudo_labels': 99, 'cumulative_coverage': 0.9907272727272727}, {'iteration': 2, 'new_pseudo_labels': 20, 'cumulative_coverage': 0.9943636363636363}, {'iteration': 3, 'new_pseudo_labels': 10, 'cumulative_coverage': 0.9961818181818182}, {'iteration': 4, 'new_pseudo_labels': 11, 'cumulative_coverage': 0.9981818181818182}, {'iteration': 5, 'new_pseudo_labels': 1, 'cumulative_coverage': 0.9983636363636363}, {'iteration': 6, 'new_pseudo_labels': 3, 'cumulative_coverage': 0.9989090909090909}, {'iteration': 7, 'new_pseudo_labels': 1, 'cumulative_coverage': 0.9990909090909091}, {'iteration': 8, 'new_pseudo_labels': 1, 'cumulative_coverage': 0.9992727272727273}, {'iteration': 9, 'new_pseudo_labels': 0, 'cumulative_coverage': 0.9